# Create Rays covering points in a pie

## Imports from package

In [ ]:
from typing import Sequence

import numpy as np
import numpy.ma as ma
from bact_bessyii_bluesky.utils.create_rays_on_grid import create_rays_on_grid, create_rays_on_turned_grid
from bact_bessyii_bluesky.utils.produce_in_between import produce_in_between
from bact_bessyii_bluesky.model.excitation_rays import Excitation, ExcitationRay, ExcitationCollection


In [ ]:
from itertools import zip_longest
import json


## General imports

In [ ]:
import matplotlib.pyplot as plt
import jsons
import pprint
from dataclasses import asdict

## Create the angles of the rays

In [ ]:
angles = [0, 90] + [angle for angle in produce_in_between(0, 90, maxdepth=4)]
angles = np.array(angles)
angles

In [ ]:

def convert_complex_ray_to_excitations(ray: Sequence[complex]) -> Sequence[Excitation]:
    return [Excitation(float(p.real), float(p.imag)) for p in ray]


def convert_complex_rays_to_excitation_rays(rays: Sequence[Sequence[complex]], angles: Sequence[float]) -> Sequence[ExcitationRay]:
    return [ExcitationRay(ray=convert_complex_ray_to_excitations(ray), target_angle=angle) for ray, angle in zip(rays,angles)]


## Input data

In [ ]:
x = np.concatenate([np.linspace(0, 3, 15, endpoint=False), np.linspace(3, 5, 20 + 1 )])
y = np.concatenate([np.linspace(0, 3, 15, endpoint=False), np.linspace(3, 5, 20 + 1)])
X, Y  = np.meshgrid(x,y)
Z_grid = X + Y * 1j
radius = 4.5
mask = np.abs(Z_grid) > radius
Zm = ma.masked_array(Z_grid, mask=mask)

r = np.linspace(0, 5, num=101)


In [ ]:
x, y

## Overview plots

### Input data 

In [ ]:
fig, ax = plt.subplots(1, 1, subplot_kw={'projection': 'polar'}, figsize=[16, 16])
ax.clear()
ax.set_thetamin(-5)
ax.set_thetamax(95)
ax.scatter(np.angle(Zm).ravel(), np.absolute(Zm).ravel(), s=.5, color='r')
ax.set_xlabel('r [kV]')

fig.subplots_adjust(bottom=0.1, right=0.85, top=0.9)

## Output data

In [ ]:
threshold = .75
rays_g, covered_points_g = create_rays_on_grid(
    Zm, angles/180.0*np.pi, r, threshold=threshold
)
rays_g = [ray for ray in rays_g if len(ray)> 2]
ec_g = ExcitationCollection(col=convert_complex_rays_to_excitation_rays(rays_g, angles))


rays, covered_points = create_rays_on_turned_grid(
    Zm, angles/180.0*np.pi, r, threshold=threshold
)

rays = [ray for ray in rays if len(ray) > 2]

ec = ExcitationCollection(col=convert_complex_rays_to_excitation_rays(rays, angles))

len(ec_g.col), len(ec_g.col)

In [ ]:

data = jsons.dump(asdict(ec))
with open("excitation_data.json", "wt") as fp:
    json.dump(data, fp)

### Plot the rays

In [ ]:
fig, ax = plt.subplots(1, 1, subplot_kw={'projection': 'polar'}, figsize=[16, 16])
ax.clear()
ax.set_thetamin(-5)
ax.set_thetamax(95)
ax.scatter(np.angle(Zm).ravel(), np.absolute(Zm).ravel(), s=2, color='r')

def ray_to_phi_radius(ray: Sequence[Excitation]):
    Z = np.array([e.horizontal + e.vertical * 1j for e in ray])
    return np.angle(Z), np.absolute(Z)

for cnt, rays in enumerate(zip_longest(ec_g.col, ec.col)):
    #if cnt>2**4:
    #    break   
    ray, ray_g  = rays
    # ray_g = None
    points = ray_to_phi_radius(ray.ray)
    line, = ax.plot(*points, marker='.', linestyle='-')
    ax.plot([ray.target_angle / 180.0*np.pi] * 2, [0, points[1][-1]], linestyle='-.', linewidth=0.25, color=line.get_color())
    ax.text(points[0][-1], points[1][-1], f"trnd {cnt+1}", color=line.get_color())

    if ray_g:
        points_g = ray_to_phi_radius(ray_g.ray)
        ax.plot(*points_g, marker='.', linestyle='--', color=line.get_color())
        ax.text(points_g[0][-1], points_g[1][-1], f"dist {cnt+1}", color=line.get_color())


### The remaining points

In [ ]:
covered_points

In [ ]:
covered_points
Z_remaining = ma.masked_array(Z_grid, mask=covered_points)

In [ ]:
fig, ax = plt.subplots(1, 1, subplot_kw={'projection': 'polar'}, figsize=[16, 16])
ax.clear()
ax.set_thetamin(-5)
ax.set_thetamax(95)
ax.scatter(np.angle(Zm).ravel(), np.absolute(Zm).ravel(), s=.1, color='b')
ax.scatter(np.angle(Z_remaining).ravel(), np.absolute(Z_remaining).ravel(), s=4, color='r')
# ax.scatter(Phi_rays.ravel(), R_rays.ravel(), s=2, color='k')

ax.set_xlabel('r [kV]')
# ax.set_rmax(1)
fig.subplots_adjust(bottom=0.1, right=0.85, top=0.9)